In [ ]:
import os
import pandas as pd
import urllib.parse
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
    
load_dotenv()
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

safe_password = urllib.parse.quote_plus(DB_PASSWORD)

db_url = f"mysql+pymysql://{DB_USER}:{safe_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

query = "SELECT * FROM AI_PROC_PREVALUE"
df = pd.read_sql(query, engine)

print(f"{df.shape}")

In [ ]:
import os
import pandas as pd
import urllib.parse
import joblib
import numpy as np
import torch
import torch.nn as nn
import seaborn as sns
import matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sklearn.metrics import recall_score, classification_report
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [ ]:
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False


# 1. 데이터 로드 및 필터링
# 두 테이블을 LOTID 기준으로 조인하여 모든 컬럼과 filepath를 가져옴
query = """
SELECT a.*, b.filepath
FROM AI_PROC_PREVALUE a
JOIN AI_VISION_DAVALUE b ON a.LOTID = b.LOTID
"""
df = pd.read_sql(query, engine)
print(f"조인 후 데이터 구조: {df.shape}")

# 실제 존재하는 이미지 파일만 남기도록 사전에 필터링
valid_rows = df['filepath'].apply(lambda x: os.path.exists(x) if pd.notnull(x) else False)
missing_count = (~valid_rows).sum()
if missing_count > 0:
    print(f"Warning: {missing_count}개의 파일을 찾을 수 없어 평가에서 제외합니다.")
df = df[valid_rows].reset_index(drop=True)
print(f"유효한 파일 필터링 후 데이터 구조: {df.shape}")

# 2. 모델 정의
# 모델 정의 및 설정
class MultimodalFusionModel(nn.Module):
    def __init__(self, num_tabular_features):
        super(MultimodalFusionModel, self).__init__()
        # Vision: ResNet18 (Feature Extractor)
        self.vision_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.vision_model.fc = nn.Identity()

        # Tabular: MLP
        self.tabular_model = nn.Sequential(
            nn.Linear(num_tabular_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Fusion Classifier
        self.classifier = nn.Sequential(
            nn.Linear(512 + 32, 128), # index 0
            nn.ReLU(),                # index 1
            nn.Dropout(0.3),          # index 2
            nn.Linear(128, 1)         # index 3
        )
    
    def forward(self, image, tab_data):
        img_features = self.vision_model(image)
        tab_features = self.tabular_model(tab_data)
        combined = torch.cat((img_features, tab_features), dim=1)
        return self.classifier(combined)

# ==========================================
# 3. 모델 및 전처리기 로드
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 가중치 로드 (OrderedDict 에러 방지용)
num_features = 28 
model = MultimodalFusionModel(num_tabular_features=num_features).to(device)
model.load_state_dict(torch.load("260402-1156-multi.pt", map_location=device))
model.eval()

scaler = joblib.load("260402-1156-multi_scaler.pkl")

# 이미지 전처리 정의 (ResNet 표준)
image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ==========================================
# 4. 데이터 전처리 (Tabular & Image)
# ==========================================
drop_cols = ['PPID', 'LOTID', 'FILENAME', 'DATIME', 'MODITIME', 'WELD_CURR_VAR', 'WELD_CURR_MAX', 'REMARK', 'filepath']

feature_cols = [col for col in df.columns if col not in drop_cols]

if 'ISERROR' in feature_cols:
    feature_cols.remove('ISERROR')

X_test_df = df[feature_cols]

X_test_df = X_test_df.select_dtypes(include=[np.number])

if X_test_df.shape[1] != 28:
    print(f"현재 피처 개수: {X_test_df.shape[1]}개.")
    print(f"현재 포함된 컬럼: {X_test_df.columns.tolist()}")

# ==========================================
# [디버깅 3] 스케일링 전 원본 데이터 통계 확인
# ==========================================
print("\n--- [디버깅 3] 스케일링 전 원본 데이터 통계 ---")
for col in X_test_df.columns:
    col_max = X_test_df[col].max()
    if col_max > 500: # 500 이상 튀는 비정상 수치가 있는지 확인
        print(f"⚠️ '{col}' 컬럼 최대값: {col_max:.2f}")

mapping = {'불량': 1, '정상': 0}
y_true = df['ISERROR'].map(mapping).values

X_test_scaled = scaler.transform(X_test_df)


# # [긴급 진단] 스케일링된 텐서의 분포 확인
# print("\n--- [스케일링 텐서 분포 확인] ---")
# print(f"최소값: {X_test_scaled.min():.2f} / 최대값: {X_test_scaled.max():.2f}")
# print(f"평균: {X_test_scaled.mean():.2f} / 표준편차: {X_test_scaled.std():.2f}")

# # ==========================================
# # [스나이퍼 디버깅] 튀는 값 추적기
# # ==========================================
# if X_test_scaled.max() > 10.0:
#     max_idx = np.unravel_index(np.argmax(X_test_scaled, axis=None), X_test_scaled.shape)
#     row_idx, col_idx = max_idx[0], max_idx[1]
#     col_name = X_test_df.columns[col_idx]
    
#     print("\n--- [🔍 스나이퍼 디버깅: 폭발한 컬럼 추적] ---")
#     print(f"문제의 컬럼: '{col_name}' (인덱스: {col_idx})")
#     print(f"▶ 해당 위치의 스케일링 된 값: {X_test_scaled[row_idx, col_idx]:.2f}")
#     print(f"▶ 해당 위치의 원본 값: {X_test_df.iloc[row_idx, col_idx]:.4f}")
#     print(f"▶ 스케일러가 기억하는 '{col_name}'의 평균: {scaler.mean_[col_idx]:.4f}")
#     print(f"▶ 스케일러가 기억하는 '{col_name}'의 표준편차: {scaler.scale_[col_idx]:.6f}")

# ==========================================
# Dataset 및 DataLoader 구성
# ==========================================
class InferenceDataset(Dataset):
    def __init__(self, filepaths, tabular_data, labels, transform):
        self.filepaths = filepaths
        self.tabular_data = tabular_data
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)
    
    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        img = Image.open(img_path).convert('RGB')
        img_tensor = self.transform(img)

        # 수치형 데이터 텐서화
        tab_tensor = torch.tensor(self.tabular_data[idx], dtype=torch.float32)

        # 라벨 반환
        label = self.labels[idx]

        return img_tensor, tab_tensor, label
    
# Dataset 및 DataLoader 생성
test_dataset = InferenceDataset(df['filepath'].values, X_test_scaled, y_true, image_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# ==========================================
# 5. 추론 진행
# ==========================================
all_preds = []
all_targets = []
all_probs = [] # [디버깅] 확률값 수집용 리스트 추가

with torch.no_grad():
    # tqdm을 사용하여 진행 상황 시각화
    for img_batch, tab_batch, target_batch in tqdm(test_loader, desc="추론 진행 중"):
        img_batch = img_batch.to(device)

        mask = torch.rand(img_batch.size(0)) < 0.10
        img_batch[mask] = 0

        tab_batch = tab_batch.to(device)
        
        logits = model(img_batch, tab_batch)
        probs = torch.sigmoid(logits)

        # [디버깅] 확률값을 1D 배열로 펼쳐서 저장
        all_probs.extend(probs.cpu().numpy().flatten())
        
        # 배치 예측값 저장
        y_pred_batch = (probs > 0.5).cpu().numpy().astype(int).flatten()
        all_preds.extend(y_pred_batch)
        all_targets.extend(target_batch.numpy())

# 리스트를 배열로 변환
final_y_true = np.array(all_targets)
final_y_pred = np.array(all_preds)
final_probs = np.array(all_probs)

# ==========================================
# [디버깅 2] 모델 예측 확률값(Probabilities) 분포 분석
# ==========================================
# print("\n--- [2. 모델 예측 확률값(Probabilities) 분포] ---")
# print(f"평균 확률: {final_probs.mean():.4f}")
# print(f"최소 확률: {final_probs.min():.4f} / 최대 확률: {final_probs.max():.4f}")

# over_threshold = (final_probs >= 0.3).sum()
# print(f"0.5 이상 (불량 예측) 건수: {over_threshold}건 / 전체 {len(final_probs)}건")

# borderline_count = ((final_probs >= 0.3) & (final_probs < 0.5)).sum()
# print(f"0.3 ~ 0.5 사이 (경계선) 건수: {borderline_count}건")

# 6. 최종 평가
print("\n--- [확률값(Probabilities) 기초 통계] ---")
print(f"평균 확률: {final_probs.mean():.4f}")
print(f"최소 확률: {final_probs.min():.4f} / 최대 확률: {final_probs.max():.4f}")

over_threshold = (final_probs >= 0.5).sum()
print(f"0.5 이상 (불량 예측) 건수: {over_threshold}건 / 전체 {len(final_probs)}건")

borderline_count = ((final_probs >= 0.3) & (final_probs < 0.5)).sum()
print(f"0.3 ~ 0.5 사이 (경계선) 건수: {borderline_count}건")

recall = recall_score(final_y_true, final_y_pred)
print(f"\n--- 최종 Recall 성능 ---")
print(f"Recall Score: {recall:.4f}")
print("\n[Detailed Report]")
print(classification_report(final_y_true, final_y_pred, target_names=['normal', 'error']))

# 7. 예측 확률 분포 히스토그램 시각화
print("\n=> 확률 분포 그래프를 생성합니다")

normal_probs = final_probs[final_y_true == 0]
error_probs = final_probs[final_y_true == 1]

plt.figure(figsize=(12, 6))

# 정상 데이터 분포
sns.histplot(normal_probs, bins=50, color='royalblue', stat='density',
             alpha=0.5, label='Normal (정상 실제값)', kde=True)

# 불량 데이터 분포
sns.histplot(error_probs, bins=50, color='tomato', stat='density',
             alpha=0.5, label='Error (불량 실제값)', kde=True)

# 현재 임계값 라인
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Current Threshold (0.5)')

plt.title('정상 vs 불량 데이터의 모델 예측 확률 분포', fontsize=16, fontweight='bold')
plt.xlabel('불량일 확률', fontsize=13)
plt.ylabel('밀도', fontsize=13)
plt.xlim(0, 1)
plt.legend(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

plt.show()